# 09. RL skeleton: формула из тезисов в коде

Здесь мы собираем **из живого `HybridScoreFollower` + `TempoTracker`** все компоненты, которые понадобятся RL-агенту, и заворачиваем их в типизированные DTO из `musictech.core.dto`.

Обучения **нет** — оно потребует ASAP-датасет, Gymnasium-env, PPO. Здесь только проверяем что:

1. Состояние `s_t` собирается из текущего стека.
2. Размерности фиксированы и не зависят от длины пьесы.
3. Награда из формулы (1) тезисов считается без проблем.

Формула (1) тезисов:

$$r_t = -|t^\text{render}_t - t^\text{perf}_t| - \lambda\, L_\text{align}(t) - \mu (a_t - a_{t-1})^2$$

Действие $a_t \in [0.5, 2.0]$ — темповый коэффициент.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from collections import deque

from musictech.core import HybridScoreFollower, TempoTracker
from musictech.core.dto import (
    AlphaSummary,
    RLAction,
    RLObservation,
    RLReward,
)
from dataset_viewer import load_performance

DATA = PROJECT_ROOT / "generated_dataset"

## 1. Из α-вектора в `AlphaSummary`

`HybridScoreFollower.hsmm.alpha` — это numpy вектор длины N (3791 для rach_solo). В политику его передавать нельзя — размер зависит от пьесы. Сжимаем в 7 чисел.

In [ ]:
def make_alpha_summary(alpha: np.ndarray, current_index: int) -> AlphaSummary:
    N = alpha.size
    n_top = min(3, N)
    top_idx_abs = np.argsort(alpha)[-n_top:][::-1]
    top_mass = float(alpha[top_idx_abs].sum())
    top_rel = tuple(int(i - current_index) for i in top_idx_abs)
    # Энтропия в натах, защищена от нулей.
    safe = alpha[alpha > 0]
    entropy = float(-(safe * np.log(safe)).sum())
    while len(top_rel) < 3:    # для пьес длиной < 3 state
        top_rel = top_rel + (0,)
    return AlphaSummary(
        max_value=float(alpha.max()),
        entropy=entropy,
        argmax_normalized=float(np.argmax(alpha)) / max(1, N - 1),
        top3_indices=top_rel[:3],
        top3_mass=top_mass,
    )

## 2. Из текущего стека — в `RLObservation`

Прогоняем `noisy.mid` через `HybridScoreFollower` + `TempoTracker` и на каждом шаге собираем `s_t`.

In [ ]:
K = 5    # окно истории темпа и эмиссионной ошибки

follower = HybridScoreFollower(str(DATA / "noisy.json"), load_tuning_profile=False)
tempo = TempoTracker(str(DATA / "noisy.json"), history_size=K)
performance = load_performance(DATA / "noisy.mid")

tempo_history = deque([1.0] * K, maxlen=K)
emission_history = deque([0.0] * K, maxlen=K)

observations: list[RLObservation] = []
for event in performance:
    idx = follower.process_event(event["pitch"], event["timestamp"])
    ratio = tempo.update(idx, event["timestamp"])

    tempo_history.append(ratio)
    emission_history.append(follower.hsmm.last_best_pitch_distance)

    obs = RLObservation(
        alpha=make_alpha_summary(follower.hsmm.alpha, follower.current_index),
        tempo_history=np.array(tempo_history, dtype=np.float64),
        emission_error_history=np.array(emission_history, dtype=np.float64),
        score_position_normalized=follower.current_index / max(1, follower.N - 1),
    )
    observations.append(obs)

print("events processed :", len(observations))
print("observation vector dim:", observations[-1].as_vector().shape)
print()
print("Last observation:")
print("  alpha:", observations[-1].alpha)
print("  tempo_history:", observations[-1].tempo_history)
print("  emission_error_history:", observations[-1].emission_error_history)
print("  position_normalized:", observations[-1].score_position_normalized)

Главное: размерность вектора фиксирована (18 = 8 alpha-features + 5 tempo + 5 emission), не зависит от того, что пьеса rach_solo на 3791 state или synthetic на 8. Это ровно то, что нужно политике RL.

## 3. Награда формула (1)

Чтобы посчитать награду, нужны:

- `t_render` — время, когда рендерер выпустил соответствующий оркестровый звук. В реальной среде это берётся из `ScoreEventDispatcher`. Здесь — заглушка.
- `t_perf` — время реального события солиста. Это `event["timestamp"]`.
- `L_align(t)` — ошибка трекера. Используем эмиссионную ошибку как прокси.
- `a_t`, `a_{t-1}` — текущее и предыдущее действие политики.

In [ ]:
def compute_reward(
    *,
    t_render: float,
    t_perf: float,
    alignment_error: float,
    action: RLAction,
    prev_action: RLAction,
    lambda_: float = 1.0,
    mu: float = 0.5,
) -> RLReward:
    sync = -abs(t_render - t_perf)
    align = -lambda_ * abs(alignment_error)
    jerk = -mu * (action.tempo_coefficient - prev_action.tempo_coefficient) ** 2
    return RLReward(
        total=sync + align + jerk,
        sync_error=sync,
        alignment_error=align,
        tempo_jerk=jerk,
    )

# Демонстрационный шаг с dummy-действиями.
prev = RLAction(tempo_coefficient=1.0)
curr = RLAction(tempo_coefficient=1.08)
reward = compute_reward(
    t_render=4.92,
    t_perf=4.95,
    alignment_error=0.12,
    action=curr,
    prev_action=prev,
)
print(reward)

## 4. Что дальше

После этой проверки понятно, что **API трекеров и DTO достаточно**, чтобы посадить сверху RL. Следующие модули писать сюда (см. `ARCHITECTURE.md` § 5.1):

- `musictech/rl/state.py` — функция `encode_state(follower, tempo) -> RLObservation` (то что в ячейке 2 этого ноутбука).
- `musictech/rl/reward.py` — функция `compute_reward(...)` (то что в ячейке 3).
- `musictech/rl/env.py` — `class ScoreFollowingEnv(gymnasium.Env)`.
- `musictech/rl/policy.py` — маленький MLP на PyTorch.
- `musictech/rl/simulator.py` — симулятор солиста с rubato (Repp 1995).
- `musictech/rl/train_bc.py` — Behavior Cloning на ASAP.
- `musictech/rl/train_ppo.py` — PPO + KL-штраф (Sequence Tutor [13]).

**Блокер для всего этого** — отсутствие реальных данных. Нужен `musictech/datasets/asap.py`, импортёр ASAP. После него BC можно обучать.